In [1]:
import pandas as pd
import numpy as np

# Reload the final 30-feature state
X_train_final = np.load('../data/X_train_final_30.npy')
X_test_final = np.load('../data/X_test_final_30.npy')
final_feature_names = np.load('../data/final_feature_names_30.npy', allow_pickle=True)

# Reload the dataframes (needed for patient_nbr and target)
df_train = pd.read_csv('../data/df_train_v2.csv')
df_test = pd.read_csv('../data/df_test_v2.csv')
y_train = df_train['target']
y_test = df_test['target']

# Verify alignment
assert len(df_train) == len(X_train_final), f"Mismatch: df_train {len(df_train)} vs X_train_final {len(X_train_final)}"

print(f"X_train_final: {X_train_final.shape}")
print(f"X_test_final:  {X_test_final.shape}")
print(f"df_train rows: {len(df_train):,}")
print(f"Positive rate: {y_train.mean()*100:.2f}%")

X_train_final: (78283, 30)
X_test_final:  (19539, 30)
df_train rows: 78,283
Positive rate: 11.33%


In [2]:
import xgboost as xgb
from sklearn.model_selection import GroupKFold, cross_val_score

# Reload state (same as before)
import pandas as pd
import numpy as np

X_train_final = np.load('../data/X_train_final_30.npy')
X_test_final = np.load('../data/X_test_final_30.npy')
final_feature_names = np.load('../data/final_feature_names_30.npy', allow_pickle=True)
df_train = pd.read_csv('../data/df_train_v2.csv')
df_test = pd.read_csv('../data/df_test_v2.csv')
y_train = df_train['target']
y_test = df_test['target']
patient_groups = df_train['patient_nbr'].values

# XGBoost with sensible defaults for an imbalanced binary classification
# scale_pos_weight handles imbalance the way class_weight='balanced' does for LR
# It's the negative-to-positive ratio in the training data
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}  (ratio of negatives to positives)")

xgb_model = xgb.XGBClassifier(
    n_estimators=200,           # number of trees — modest default
    max_depth=4,                # tree depth — modest, prevents overfitting
    learning_rate=0.1,          # standard
    scale_pos_weight=scale_pos_weight,  # handle imbalance
    eval_metric='auc',          # what we're optimizing
    random_state=42,
    n_jobs=-1                   # parallel
)

# Run the same patient-grouped 5-fold CV
group_kfold = GroupKFold(n_splits=5)
xgb_cv_scores = cross_val_score(
    xgb_model,
    X_train_final,
    y_train,
    groups=patient_groups,
    cv=group_kfold,
    scoring='roc_auc',
    n_jobs=-1
)

print(f"\nXGBoost CV AUC scores: {xgb_cv_scores}")
print(f"Mean AUC: {xgb_cv_scores.mean():.4f}")
print(f"Std Dev:  {xgb_cv_scores.std():.4f}")
print(f"95% CI:   [{xgb_cv_scores.mean() - 2*xgb_cv_scores.std():.4f}, {xgb_cv_scores.mean() + 2*xgb_cv_scores.std():.4f}]")

# Comparison table
print(f"\n--- LR vs XGBoost (5-fold patient-grouped CV) ---")
print(f"{'Model':<15} {'Mean AUC':<12} {'Std Dev':<10} {'95% CI':<25}")
print(f"{'-'*70}")
print(f"{'LR (current)':<15} {0.6607:<12.4f} {0.0025:<10.4f} {'[0.656, 0.666]':<25}")
print(f"{'XGBoost':<15} {xgb_cv_scores.mean():<12.4f} {xgb_cv_scores.std():<10.4f} {f'[{xgb_cv_scores.mean() - 2*xgb_cv_scores.std():.4f}, {xgb_cv_scores.mean() + 2*xgb_cv_scores.std():.4f}]':<25}")

scale_pos_weight: 7.83  (ratio of negatives to positives)

XGBoost CV AUC scores: [0.65984708 0.66780085 0.661783   0.66811412 0.66583043]
Mean AUC: 0.6647
Std Dev:  0.0033
95% CI:   [0.6581, 0.6713]

--- LR vs XGBoost (5-fold patient-grouped CV) ---
Model           Mean AUC     Std Dev    95% CI                   
----------------------------------------------------------------------
LR (current)    0.6607       0.0025     [0.656, 0.666]           
XGBoost         0.6647       0.0033     [0.6581, 0.6713]         


In [3]:
# Save the XGBoost results for the comparison record
import joblib

xgb_model.fit(X_train_final, y_train)
joblib.dump(xgb_model, '../data/xgboost_model_30features.pkl')

# Save the comparison
comparison_record = pd.DataFrame({
    'Model': ['LR (L2, class_weight=balanced)', 'XGBoost (defaults, scale_pos_weight)'],
    'CV_mean_AUC': [0.6607, 0.6647],
    'CV_std': [0.0025, 0.0033],
    'CI_low': [0.6557, 0.6581],
    'CI_high': [0.6657, 0.6713],
    'Decision': ['Selected as final model', 'Tested - marginal improvement within noise']
})
comparison_record.to_csv('../data/lr_vs_xgb_comparison.csv', index=False)
print("Saved XGBoost model and LR-vs-XGBoost comparison")

Saved XGBoost model and LR-vs-XGBoost comparison


# Non-Linear Model Comparison

## Context

Cross-validation in `08_cross_validation.ipynb` confirmed the LR baseline 
achieves AUC 0.6607 ± 0.0025 with high stability across patient-grouped 
folds. The tight standard deviation suggested the linear model has 
converged on its best representation of the data and that the AUC 
ceiling may be real.

This notebook tests whether a non-linear model (XGBoost) can break that 
ceiling.

## Methodology

XGBoost trained on the same 30-feature set with sensible defaults:
- 200 trees, max_depth=4, learning_rate=0.1
- `scale_pos_weight` = 7.83 to handle 11.3% positive class
- Same patient-grouped 5-fold CV as LR

## Results

| Model | Mean AUC | Std Dev | 95% CI |
|-------|----------|---------|--------|
| LR (current) | 0.6607 | 0.0025 | [0.656, 0.666] |
| XGBoost | 0.6647 | 0.0033 | [0.658, 0.671] |

XGBoost's CI overlaps substantially with LR's; the 0.004 mean difference 
is smaller than fold-to-fold variance.

## Interpretation

The marginal XGBoost improvement and overlapping confidence intervals 
indicate that **non-linearity does not substantially improve predictions 
on this problem**. Two compatible explanations:

1. **The features lack strong interactions.** Tree models gain their 
   advantage primarily by capturing feature interactions automatically. 
   If LR with engineered linear effects is within 0.01 AUC of XGBoost 
   with full interaction capacity, the interactions in this feature 
   space are weak.

2. **The data has a noise floor.** Administrative healthcare data 
   captures only a subset of factors that drive 30-day readmission. 
   Unmeasured factors (medication adherence, social support, post-discharge 
   follow-up compliance) likely set an absolute ceiling on predictability.

Published literature on the Diabetes 130 dataset typically reports AUC 
in the 0.65–0.72 range, supporting interpretation (2).

## Decision

**Retain the 30-feature L2 logistic regression as the final model.**

Three reasons:
1. XGBoost's improvement is within fold-to-fold noise (cannot claim 
   statistical superiority)
2. Logistic regression's coefficients are interpretable; XGBoost's 
   ensemble decisions require SHAP for interpretation
3. The remaining performance ceiling reflects data-level constraints, 
   not model-class constraints, so further model complexity is unlikely 
   to help

Subsequent work focuses on:
- Interpretability via coefficient analysis and SHAP
- Fairness audit across demographic groups  
- Honest deployment-aware writeup
- Interactive risk-scoring dashboard